Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
    OptunaOptimizer,
    smote_objective,
    pure_smote_objective,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'optuna-opt-mean-smote_default-model'
add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 400

save_model_and_metrics = True
metrics_file = 'metrics_modelling4_smote_mean_opt.xlsx'

## Optimization functions
Optuna will not be used further, except first try, it is enough to conduct grid search

In [5]:
def optimize_smote_optuna(
    target:str,
    estimator:object,
    estimator_params:dict,
    n_trials:int=n_trials,
    step_scoring_average:str=step_scoring_average,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    seed:int=seed,
):
    """
    Optimize the SMOTE parameters for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        n_trials: The number of trials for optimization. Defaults to n_trials.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    
    smote_params = smote_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )

    opt = OptunaOptimizer(
        objective=partial(smote_objective, ml_pipe=ml_pipe),
        study_name="smote_study",
        direction="maximize",
        seed=seed,
    )

    opt.optimize(n_trials=n_trials)

    best_params = opt.study.best_params
    print('best_params')
    display(best_params)

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params={**smote_params, **best_params},
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:47:50,524] A new study created in memory with name: smote_study
[I 2025-04-16 00:47:50,601] Trial 0 finished with value: 0.808631916777597 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:47:50,670] Trial 1 finished with value: 0.8033759141203999 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:47:50,740] Trial 2 finished with value: 0.8082580539469134 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:47:50,807] Trial 3 finished with value: 0.8086807655106563 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8086807655106563.
[I 2025-04-16 00:47:50,877] Trial 4 finished with value: 0.8069288014407415 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.885031
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.871517


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smote_optuna-opt-mean-smote_default-model


In [7]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:48:25,402] A new study created in memory with name: smote_study
[I 2025-04-16 00:48:25,483] Trial 0 finished with value: 0.808631916777597 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:48:25,562] Trial 1 finished with value: 0.8033759141203999 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:48:25,637] Trial 2 finished with value: 0.8082580539469134 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:48:25,712] Trial 3 finished with value: 0.8086807655106563 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8086807655106563.
[I 2025-04-16 00:48:25,785] Trial 4 finished with value: 0.8069288014407415 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.885031
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.871517


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smote_optuna-opt-mean-smote_default-model


Full grid search can find better solution, than Optuna with small number of iterations - like 40! It appears in some Random_seed

In [8]:
estimator = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:48:59,058] A new study created in memory with name: smote_study
[I 2025-04-16 00:48:59,128] Trial 0 finished with value: 0.7953914760068441 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.7953914760068441.
[I 2025-04-16 00:48:59,196] Trial 1 finished with value: 0.7893690960867744 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.7953914760068441.
[I 2025-04-16 00:48:59,260] Trial 2 finished with value: 0.7927422008145589 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.7953914760068441.
[I 2025-04-16 00:48:59,327] Trial 3 finished with value: 0.7972483300592221 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.7972483300592221.
[I 2025-04-16 00:48:59,390] Trial 4 finished with value: 0.7927358172501057 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smote_optu...
holdout_test_f1_macro,0.802991
holdout_test_accuracy_balanced,0.85
holdout_test_roc_auc,0.954545
holdout_test_f1,0.734694
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.93956


Model saved in ../results/models_modelling4/LogisticRegression_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# KNClassifier

In [9]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:49:28,734] A new study created in memory with name: smote_study
[I 2025-04-16 00:49:28,859] Trial 0 finished with value: 0.839000057333054 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:49:28,963] Trial 1 finished with value: 0.8366744957233221 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:49:29,106] Trial 2 finished with value: 0.8275188914323743 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:49:29,210] Trial 3 finished with value: 0.8339107074490688 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:49:29,314] Trial 4 finished with value: 0.826598877826172 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is tr

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smote_optuna-op...
holdout_test_f1_macro,0.829027
holdout_test_accuracy_balanced,0.834491
holdout_test_roc_auc,0.891204
holdout_test_f1,0.87234
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.846377
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.929323


Model saved in ../results/models_modelling4/KNeighborsClassifier_splashing_smote_optuna-opt-mean-smote_default-model


In [10]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:50:13,580] A new study created in memory with name: smote_study
[I 2025-04-16 00:50:13,691] Trial 0 finished with value: 0.8165889152726052 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8165889152726052.
[I 2025-04-16 00:50:13,803] Trial 1 finished with value: 0.8473500834496152 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8473500834496152.
[I 2025-04-16 00:50:13,901] Trial 2 finished with value: 0.8312963436962847 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8473500834496152.
[I 2025-04-16 00:50:13,997] Trial 3 finished with value: 0.8152523468202186 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8473500834496152.
[I 2025-04-16 00:50:14,118] Trial 4 finished with value: 0.8055981156379303 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smote_op...
holdout_test_f1_macro,0.825397
holdout_test_accuracy_balanced,0.852273
holdout_test_roc_auc,0.943636
holdout_test_f1,0.755556
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.841642
cv_test_accuracy_balanced_median,0.897436
cv_test_roc_auc_median,0.951282


Model saved in ../results/models_modelling4/KNeighborsClassifier_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# SVC

In [11]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:50:54,430] A new study created in memory with name: smote_study
[I 2025-04-16 00:50:54,549] Trial 0 finished with value: 0.8617852500049572 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8617852500049572.
[I 2025-04-16 00:50:54,660] Trial 1 finished with value: 0.8644198887587763 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8644198887587763.
[I 2025-04-16 00:50:54,764] Trial 2 finished with value: 0.8550552362111962 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8644198887587763.
[I 2025-04-16 00:50:54,885] Trial 3 finished with value: 0.8529924934958182 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8644198887587763.
[I 2025-04-16 00:50:55,004] Trial 4 finished with value: 0.8716691814330063 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smote_optuna-opt-mean-smote_defa...
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.881944
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.870743
cv_test_roc_auc_median,0.925697


Model saved in ../results/models_modelling4/SVC_splashing_smote_optuna-opt-mean-smote_default-model


In [12]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:51:45,809] A new study created in memory with name: smote_study
[I 2025-04-16 00:51:45,922] Trial 0 finished with value: 0.8317057142869242 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8317057142869242.
[I 2025-04-16 00:51:46,026] Trial 1 finished with value: 0.8346658965165396 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:51:46,136] Trial 2 finished with value: 0.8331099974208901 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:51:46,245] Trial 3 finished with value: 0.831839385795292 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:51:46,353] Trial 4 finished with value: 0.8150515241534569 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smote_optuna-opt-mean-smo...
holdout_test_f1_macro,0.857143
holdout_test_accuracy_balanced,0.886364
holdout_test_roc_auc,0.950909
holdout_test_f1,0.8
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.948718


Model saved in ../results/models_modelling4/SVC_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# CatBoost

In [13]:
estimator = CatBoostClassifier
target = 'splashing'
estimator_params = {
    'verbose': False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:52:31,252] A new study created in memory with name: smote_study
[I 2025-04-16 00:52:36,543] Trial 0 finished with value: 0.8868392980653509 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8868392980653509.
[I 2025-04-16 00:52:41,545] Trial 1 finished with value: 0.8825840948565951 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8868392980653509.
[I 2025-04-16 00:52:46,260] Trial 2 finished with value: 0.8968166188226805 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8968166188226805.
[I 2025-04-16 00:52:51,433] Trial 3 finished with value: 0.8834258532725957 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 2 with value: 0.8968166188226805.
[I 2025-04-16 00:52:56,476] Trial 4 finished with value: 0.8871726599131404 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.863609
holdout_test_accuracy_balanced,0.849537
holdout_test_roc_auc,0.95679
holdout_test_f1,0.910891
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.881696
cv_test_accuracy_balanced_median,0.900155
cv_test_roc_auc_median,0.952012


Model saved in ../results/models_modelling4/CatBoostClassifier_splashing_smote_optuna-opt-mean-smote_default-model


In [14]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 01:25:45,740] A new study created in memory with name: smote_study
[I 2025-04-16 01:25:51,317] Trial 0 finished with value: 0.8515510297419553 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8515510297419553.
[I 2025-04-16 01:25:56,836] Trial 1 finished with value: 0.8365369752968795 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8515510297419553.
[I 2025-04-16 01:26:02,020] Trial 2 finished with value: 0.853562425722252 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.853562425722252.
[I 2025-04-16 01:26:07,566] Trial 3 finished with value: 0.8496138363462047 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 2 with value: 0.853562425722252.
[I 2025-04-16 01:26:13,078] Trial 4 finished with value: 0.8504222020545641 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smote_optu...
holdout_test_f1_macro,0.918496
holdout_test_accuracy_balanced,0.938636
holdout_test_roc_auc,0.984545
holdout_test_f1,0.883721
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.887363
cv_test_roc_auc_median,0.97265


Model saved in ../results/models_modelling4/CatBoostClassifier_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# XGBoost

In [15]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:01:17,771] A new study created in memory with name: smote_study
[I 2025-04-16 02:01:19,752] Trial 0 finished with value: 0.850973610215128 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.850973610215128.
[I 2025-04-16 02:01:21,313] Trial 1 finished with value: 0.8792709610761528 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8792709610761528.
[I 2025-04-16 02:01:22,859] Trial 2 finished with value: 0.8737949693941839 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8792709610761528.
[I 2025-04-16 02:01:24,436] Trial 3 finished with value: 0.888533166234959 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.888533166234959.
[I 2025-04-16 02:01:26,001] Trial 4 finished with value: 0.8864529148094934 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smote_optuna-opt-mean-...
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.94483
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.95356


Model saved in ../results/models_modelling4/XGBClassifier_splashing_smote_optuna-opt-mean-smote_default-model


In [16]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:12:12,987] A new study created in memory with name: smote_study
[I 2025-04-16 02:12:14,587] Trial 0 finished with value: 0.8239303192151926 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8239303192151926.
[I 2025-04-16 02:12:16,104] Trial 1 finished with value: 0.83412563873452 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.83412563873452.
[I 2025-04-16 02:12:17,659] Trial 2 finished with value: 0.8324181440883308 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.83412563873452.
[I 2025-04-16 02:12:19,246] Trial 3 finished with value: 0.8419022858785005 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8419022858785005.
[I 2025-04-16 02:12:21,006] Trial 4 finished with value: 0.8374503834259999 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is tr

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smote_optuna-op...
holdout_test_f1_macro,0.918496
holdout_test_accuracy_balanced,0.938636
holdout_test_roc_auc,0.990909
holdout_test_f1,0.883721
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.962393


Model saved in ../results/models_modelling4/XGBClassifier_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# AdaBoost

In [17]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:19:47,200] A new study created in memory with name: smote_study
[I 2025-04-16 02:19:47,473] Trial 0 finished with value: 0.8640537066058273 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8640537066058273.
[I 2025-04-16 02:19:47,735] Trial 1 finished with value: 0.8437932211095193 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8640537066058273.
[I 2025-04-16 02:19:47,994] Trial 2 finished with value: 0.8271081497714823 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8640537066058273.
[I 2025-04-16 02:19:48,260] Trial 3 finished with value: 0.8276516517605865 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.8640537066058273.
[I 2025-04-16 02:19:48,529] Trial 4 finished with value: 0.8616016835043357 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.755981
holdout_test_accuracy_balanced,0.758102
holdout_test_roc_auc,0.846065
holdout_test_f1,0.821053
holdout_test_accuracy,0.773333
cv_test_f1_macro_median,0.805718
cv_test_accuracy_balanced_median,0.829721
cv_test_roc_auc_median,0.885449


Model saved in ../results/models_modelling4/AdaBoostClassifier_splashing_smote_optuna-opt-mean-smote_default-model


In [18]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:21:36,822] A new study created in memory with name: smote_study
[I 2025-04-16 02:21:37,090] Trial 0 finished with value: 0.8411554125130493 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8411554125130493.
[I 2025-04-16 02:21:37,349] Trial 1 finished with value: 0.8137546557969431 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8411554125130493.
[I 2025-04-16 02:21:37,606] Trial 2 finished with value: 0.8252131718939804 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8411554125130493.
[I 2025-04-16 02:21:37,872] Trial 3 finished with value: 0.8330161401074437 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.8411554125130493.
[I 2025-04-16 02:21:38,136] Trial 4 finished with value: 0.7949676755822634 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smote_optu...
holdout_test_f1_macro,0.780518
holdout_test_accuracy_balanced,0.809091
holdout_test_roc_auc,0.930455
holdout_test_f1,0.695652
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.815385
cv_test_accuracy_balanced_median,0.828755
cv_test_roc_auc_median,0.912088


Model saved in ../results/models_modelling4/AdaBoostClassifier_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# Random Forest

In [19]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:23:24,061] A new study created in memory with name: smote_study
[I 2025-04-16 02:23:24,541] Trial 0 finished with value: 0.8620945466055197 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8620945466055197.
[I 2025-04-16 02:23:25,007] Trial 1 finished with value: 0.8629914840212102 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8629914840212102.
[I 2025-04-16 02:23:25,459] Trial 2 finished with value: 0.8686836798098685 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8686836798098685.
[I 2025-04-16 02:23:25,923] Trial 3 finished with value: 0.8811047094343298 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8811047094343298.
[I 2025-04-16 02:23:26,389] Trial 4 finished with value: 0.8729282104847041 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smote_optuna-...
holdout_test_f1_macro,0.82
holdout_test_accuracy_balanced,0.810185
holdout_test_roc_auc,0.953318
holdout_test_f1,0.88
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.87381
cv_test_accuracy_balanced_median,0.87381
cv_test_roc_auc_median,0.946032


Model saved in ../results/models_modelling4/RandomForestClassifier_splashing_smote_optuna-opt-mean-smote_default-model


In [20]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:26:44,964] A new study created in memory with name: smote_study
[I 2025-04-16 02:26:45,540] Trial 0 finished with value: 0.8404884202789548 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8404884202789548.
[I 2025-04-16 02:26:46,040] Trial 1 finished with value: 0.8279257550100352 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8404884202789548.
[I 2025-04-16 02:26:46,517] Trial 2 finished with value: 0.8396656388231067 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8404884202789548.
[I 2025-04-16 02:26:47,048] Trial 3 finished with value: 0.8353574430162406 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.8404884202789548.
[I 2025-04-16 02:26:47,577] Trial 4 finished with value: 0.8432365084612183 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smote_...
holdout_test_f1_macro,0.903516
holdout_test_accuracy_balanced,0.929545
holdout_test_roc_auc,0.980455
holdout_test_f1,0.863636
holdout_test_accuracy,0.92
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.954701


Model saved in ../results/models_modelling4/RandomForestClassifier_no_fragmentation_smote_optuna-opt-mean-smote_default-model


# LightGBM

In [21]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 02:30:10,764] A new study created in memory with name: smote_study
[I 2025-04-16 02:30:13,450] Trial 0 finished with value: 0.8937668325736822 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8937668325736822.
[I 2025-04-16 02:30:16,022] Trial 1 finished with value: 0.8758155858900383 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8937668325736822.
[I 2025-04-16 02:30:18,422] Trial 2 finished with value: 0.8701472818088798 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.8937668325736822.
[I 2025-04-16 02:30:21,123] Trial 3 finished with value: 0.8760492580553016 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.8937668325736822.
[I 2025-04-16 02:30:23,652] Trial 4 finished with value: 0.8945019436033209 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smote_optuna-opt-mean...
holdout_test_f1_macro,0.852826
holdout_test_accuracy_balanced,0.847222
holdout_test_roc_auc,0.949846
holdout_test_f1,0.897959
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.858018
cv_test_accuracy_balanced_median,0.862229
cv_test_roc_auc_median,0.953968


Model saved in ../results/models_modelling4/LGBMClassifier_splashing_smote_optuna-opt-mean-smote_default-model


In [22]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 02:47:17,015] A new study created in memory with name: smote_study
[I 2025-04-16 02:47:20,060] Trial 0 finished with value: 0.8422797523462326 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8422797523462326.
[I 2025-04-16 02:47:22,973] Trial 1 finished with value: 0.8599197691887189 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8599197691887189.
[I 2025-04-16 02:47:25,655] Trial 2 finished with value: 0.8342135201627616 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8599197691887189.
[I 2025-04-16 02:47:28,826] Trial 3 finished with value: 0.8531695449663894 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8599197691887189.
[I 2025-04-16 02:47:31,815] Trial 4 finished with value: 0.8639931824265202 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smote_optuna-o...
holdout_test_f1_macro,0.913375
holdout_test_accuracy_balanced,0.906818
holdout_test_roc_auc,0.989091
holdout_test_f1,0.871795
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.880037
cv_test_roc_auc_median,0.97094


Model saved in ../results/models_modelling4/LGBMClassifier_no_fragmentation_smote_optuna-opt-mean-smote_default-model
